# Modelo de riesgo de pérdida · Rentabilidad y política de descuentos
Grupo 3 · Inteligencia de Negocios. Cuaderno de apoyo: reproduce las cifras de la página del modelo.

**Pregunta:** ¿qué líneas de venta terminan en pérdida (ganancia < 0) y qué combinación de descuento, producto y mercado lo anticipa?
**Método:** entrenar con 2011–2013 y probar con 2014. La ganancia nunca entra como dato de entrada.

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 1. Cargar el CSV
Súbelo con el botón de archivos de Colab (carpeta a la izquierda) y escribe su nombre en `RUTA`.

Los nombres de producto traen comillas dentro del texto (por ejemplo `TOPS "Important Message" Pads`) y `pd.read_csv` falla en esas filas. Como todos los campos van entre comillas, se lee separando por `","`.

In [ ]:
RUTA = 'datos_rentabilidad.csv'

def cargar(ruta):
    filas = []
    with open(ruta, encoding='utf-8-sig') as f:
        cab = f.readline().rstrip('\r\n')[1:-1].split('","')
        for linea in f:
            p = linea.rstrip('\r\n')[1:-1].split('","')
            assert len(p) == len(cab), linea[:80]
            filas.append(p)
    df = pd.DataFrame(filas, columns=cab)
    for c in ['VentaKey','Anio','Trimestre','Mes','Cantidad','EsPerdida']: df[c] = df[c].astype(int)
    for c in ['Ventas','Descuento','CostoEnvio','Ganancia']: df[c] = df[c].astype(float)
    return df

df = cargar(RUTA)
# Controles: deben coincidir con el DataMart
print(df.shape)                                    # (51290, 22)
print(round(df.Ventas.sum(), 2), round(df.Ganancia.sum(), 2))   # 12642500.86  1467456.7
print(round(df.EsPerdida.mean()*100, 2), df.CodigoPedido.nunique())  # 24.46  25035

## 2. Separar por tiempo

In [ ]:
tr = df[df.Anio <= 2013]
te = df[df.Anio == 2014]
print(len(tr), len(te), round(tr.EsPerdida.mean(), 4), round(te.EsPerdida.mean(), 4))

## 3. Regla base y modelos (todas las líneas)

In [ ]:
cat = ['Subcategoria','Mercado','Segmento','ModoEnvio','Prioridad']
num = ['Descuento','Cantidad','Trimestre']
def X(d, cols=None):
    x = pd.get_dummies(d[cat], dtype=float).join(d[num])
    return x if cols is None else x.reindex(columns=cols, fill_value=0)
Xtr = X(tr); Xte = X(te, Xtr.columns)

def metricas(nombre, p, pred):
    y = te.EsPerdida
    print(f'{nombre:<28} exactitud {accuracy_score(y,pred):.3f}  precisión {precision_score(y,pred):.3f}  '
          f'recall {recall_score(y,pred):.3f}  F1 {f1_score(y,pred):.3f}  AUC {roc_auc_score(y,p):.3f}')

metricas('Regla base (desc. > 20 %)', te.Descuento, (te.Descuento > .2).astype(int))

arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=150, random_state=0).fit(Xtr, tr.EsPerdida)
p = arbol.predict_proba(Xte)[:,1]; metricas('Árbol de decisión (prof. 4)', p, (p >= .5).astype(int))

logit = LogisticRegression(max_iter=3000).fit(Xtr, tr.EsPerdida)
p = logit.predict_proba(Xte)[:,1]; metricas('Regresión logística', p, (p >= .5).astype(int))

print(export_text(arbol, feature_names=list(Xtr.columns)))

## 4. Modelo de tres variables (descuento, subcategoría, mercado)
Es monótono en el descuento: a mayor descuento, nunca baja la probabilidad de pérdida. Es el que usa el simulador de la página.

In [ ]:
f3 = ['Descuento','Subcategoria','Mercado']
subs = sorted(df.Subcategoria.unique()); mkts = sorted(df.Mercado.unique())
def prep(d):
    x = d[f3].copy()
    x['Subcategoria'] = pd.Categorical(x.Subcategoria, categories=subs)
    x['Mercado'] = pd.Categorical(x.Mercado, categories=mkts)
    return x
def gbm():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=.08, max_iter=250,
        categorical_features=[1,2], monotonic_cst=[1,0,0], random_state=0)

g = gbm().fit(prep(tr), tr.EsPerdida)
p = g.predict_proba(prep(te))[:,1]; metricas('Modelo de 3 variables', p, (p >= .5).astype(int))

# Calibración: ¿la probabilidad anunciada coincide con lo que pasó en 2014?
cal = pd.DataFrame({'p': p, 'real': te.EsPerdida.values})
cal['tramo'] = pd.cut(cal.p, [0,.1,.2,.3,.5,.7,.9,1.0001], right=False)
print(cal.groupby('tramo', observed=True).agg(lineas=('p','size'), anunciado=('p','mean'), real=('real','mean')).round(3))

## 5. El rango permitido: descuento de 1 % a 20 %
Aquí está el valor del modelo. Con todas las líneas el descuento domina; dentro del tope, subcategoría y mercado separan el riesgo.

In [ ]:
sub_tr = tr[(tr.Descuento > 0) & (tr.Descuento <= .2)]
sub_te = te[(te.Descuento > 0) & (te.Descuento <= .2)]
print('líneas con pérdida en el rango 1–20 %:', round(df[(df.Descuento>0)&(df.Descuento<=.2)].EsPerdida.mean(), 3))

g2 = gbm().fit(prep(sub_tr), sub_tr.EsPerdida)
q = pd.Series(g2.predict_proba(prep(sub_te))[:,1], index=sub_te.index)
print('AUC en el rango permitido (2014):', round(roc_auc_score(sub_te.EsPerdida, q), 3))
top = q >= q.quantile(.8)
print('20 % más riesgoso pierde:', round(sub_te.EsPerdida[top].mean(), 3), '| el resto:', round(sub_te.EsPerdida[~top].mean(), 3))

# Mapa de calor: % de pérdida por subcategoría y mercado (celdas con >= 30 líneas)
w = df[(df.Descuento > 0) & (df.Descuento <= .2)]
t = w.pivot_table(index='Subcategoria', columns='Mercado', values='EsPerdida', aggfunc='mean')
n = w.pivot_table(index='Subcategoria', columns='Mercado', values='EsPerdida', aggfunc='size')
print((t.where(n >= 30) * 100).round(0))

## 6. Descuento máximo sugerido y prueba hacia atrás con 2014
Regla: el descuento más alto con margen ≥ 5 %, siempre que todos los niveles menores (con ≥ 30 líneas) también lo cumplan.

In [ ]:
df['Nivel'] = df.Descuento.round(2)
tr = df[df.Anio <= 2013]; te = df[df.Anio == 2014]

def limite(d, min_lineas=30, margen_min=.05):
    cap = 0.0
    g = d.groupby('Nivel').agg(n=('Ganancia','size'), gan=('Ganancia','sum'), ven=('Ventas','sum')).sort_index()
    for nivel, r in g.iterrows():
        if nivel == 0 or r.n < min_lineas: continue
        if r.gan / r.ven >= margen_min: cap = nivel
        else: break
    return cap

print('Límite global con todos los datos:', limite(df))
print({s: limite(df[df.Subcategoria == s]) for s in subs})

caps_tr = {s: limite(tr[tr.Subcategoria == s]) for s in subs}   # solo 2011–2013
def prueba(nombre, tope):
    sobre = te.Descuento > te.apply(tope, axis=1) + 1e-9
    dentro = te[~sobre]
    print(f'{nombre:<24} sobre el tope: {sobre.sum():>5} líneas, ganancia {te.Ganancia[sobre].sum():>10,.0f} | '
          f'quedan {len(dentro):>6}, ganancia {dentro.Ganancia.sum():>9,.0f}, margen {dentro.Ganancia.sum()/dentro.Ventas.sum():.3f}, '
          f'pierden {dentro.EsPerdida.mean():.3f}')

prueba('Tope 20 %', lambda r: .20)
prueba('Tope 25 %', lambda r: .25)
prueba('Tope 30 %', lambda r: .30)
prueba('Tope por subcategoría', lambda r: caps_tr[r.Subcategoria])

## 7. Modelo ampliado y modelo de margen
Se suman dos datos que se conocen al momento de vender: el precio de lista y el margen habitual del producto cuando se vende sin descuento (calculado solo con los años de entrenamiento). El modelo de margen estima cuánta ganancia deja cada venta.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
df = cargar(RUTA)
df['PrecioLista']=df.Ventas/(df.Cantidad*(1-df.Descuento))
df['Margen']=df.Ganancia/df.Ventas
CATS=['Subcategoria','Mercado','Segmento','ModoEnvio','Prioridad']
def prodfeat(hist):
    z=hist[hist.Descuento==0]
    pm=z.groupby('NombreProducto').apply(lambda d:d.Ganancia.sum()/d.Ventas.sum(),include_groups=False)
    pn=z.groupby('NombreProducto').size()
    sm=z.groupby('Subcategoria').apply(lambda d:d.Ganancia.sum()/d.Ventas.sum(),include_groups=False)
    return pm,pn,sm
def addf(d,pm,pn,sm):
    d=d.copy();d['MargenBaseProd']=d.NombreProducto.map(pm);d['ConHist']=d.NombreProducto.map(pn).fillna(0)
    d['MargenBaseProd']=d.MargenBaseProd.fillna(d.Subcategoria.map(sm));return d
FEATS=['Descuento','Cantidad','Trimestre','PrecioLista','MargenBaseProd','ConHist']+CATS
def X(d):
    x=d[FEATS].copy()
    for c in CATS:x[c]=pd.Categorical(x[c],categories=sorted(df[c].unique()))
    return x
cidx=[FEATS.index(c) for c in CATS]
def clf():return HistGradientBoostingClassifier(max_iter=400,learning_rate=.06,categorical_features=cidx,random_state=0)
def reg():return HistGradientBoostingRegressor(max_iter=500,learning_rate=.06,categorical_features=cidx,loss='absolute_error',random_state=0)

## 8. Validación con 2014 y proyección 2015
Primero se valida con 2014 (entrenando con 2011–2013). Luego se entrena con 2011–2014 y se aplica a la cartera 2015: la mezcla de ventas de 2014 escalada al pronóstico mensual de Excel (PRONOSTICO.ETS), sin tope y con tope de 20 %.

In [ ]:
from sklearn.metrics import r2_score
def regq():return HistGradientBoostingRegressor(max_iter=500,learning_rate=.06,categorical_features=cidx,random_state=0)
OUT={}
# ---------- validación 2014
tr=df[df.Anio<=2013];te=df[df.Anio==2014]
pm,pn,sm=prodfeat(tr);trf=addf(tr,pm,pn,sm);tef=addf(te,pm,pn,sm)
c=clf().fit(X(trf),trf.EsPerdida);p=c.predict_proba(X(tef))[:,1];y=te.EsPerdida
OUT['amp']=dict(acc=accuracy_score(y,p>=.5),prec=precision_score(y,p>=.5),rec=recall_score(y,p>=.5),f1=float(2*precision_score(y,p>=.5)*recall_score(y,p>=.5)/(precision_score(y,p>=.5)+recall_score(y,p>=.5))),auc=roc_auc_score(y,p))
r=regq().fit(X(trf),trf.Margen);mp=r.predict(X(tef))
OUT['val']=dict(r2_margin=r2_score(te.Margen,mp),r2_profit=r2_score(te.Ganancia,mp*te.Ventas),pred_total=float((mp*te.Ventas).sum()),real_total=float(te.Ganancia.sum()),
  pred_loss_lines=float(p.sum()),real_loss_lines=int(y.sum()))
mm=pd.DataFrame({'Mes':te.Mes,'real':te.Ganancia,'pred':mp*te.Ventas}).groupby('Mes').sum()
OUT['val_month']=dict(real=mm.real.round(0).tolist(),pred=mm.pred.round(0).tolist())
# respuesta al tope en 2014: modelo vs fórmula SSAS
def capped(d,t):
    d=d.copy();d0=d.Descuento.copy();d['Descuento']=np.minimum(d0,t);d['VentasT']=d.Ventas/(1-d0)*(1-d.Descuento);return d
tc=capped(tef,.2)
OUT['val_tope']=dict(model=float((r.predict(X(tc))*tc.VentasT).sum()),ssas=float((tc.VentasT-(te.Ventas-te.Ganancia)).sum()))
# ---------- modelos finales 2011–2014 y cartera 2015
pm,pn,sm=prodfeat(df);dff=addf(df,pm,pn,sm)
cF=clf().fit(X(dff),dff.EsPerdida);rF=regq().fit(X(dff),dff.Margen)
FC=[334656.01,300155.38,346137.25,333959.32,397288.75,456262.69,351274.73,477782.00,513821.72,451821.70,537629.21,560611.20]
LO=[276431.99,235032.90,274756.29,256803.95,314741.35,368634.97,258826.97,380735.65,412368.09,346128.12,427843.59,446865.47]
HI=[392880.02,365277.87,417518.20,411114.68,479836.15,543890.41,443722.49,574828.34,615275.35,557515.29,647414.82,674356.93]
c14=dff[dff.Anio==2014].copy()
s14=c14.groupby('Mes').Ventas.sum()
c14['w']=c14.Mes.map(lambda m:FC[m-1]/s14[m])
def scen(d,t=None):
    if t is not None: d=capped(d,t); v=d.VentasT
    else: v=d.Ventas
    marg=rF.predict(X(d));pl=cF.predict_proba(X(d))[:,1]
    g=d.w*v*marg
    return d.assign(V=d.w*v,G=g,PL=d.w*pl)
s0=scen(c14);s1=scen(c14,.2)
ssas=(c14.w*(capped(c14,.2).VentasT-(c14.Ventas-c14.Ganancia)))
OUT['y2015']=dict(lines=float(c14.w.sum()),ventas_fc=sum(FC),
  sin=dict(ventas=float(s0.V.sum()),gan=float(s0.G.sum()),loss_lines=float(s0.PL.sum())),
  tope=dict(ventas=float(s1.V.sum()),gan=float(s1.G.sum()),loss_lines=float(s1.PL.sum()),ssas=float(ssas.sum())))
OUT['y2015_month']=dict(fc=FC,lo=LO,hi=HI,gan_sin=s0.groupby('Mes').G.sum().round(0).tolist(),gan_tope=s1.groupby('Mes').G.sum().round(0).tolist(),
  loss_sin=(s0.groupby('Mes').PL.sum()/c14.groupby('Mes').w.sum()).round(4).tolist(),loss_tope=(s1.groupby('Mes').PL.sum()/c14.groupby('Mes').w.sum()).round(4).tolist())
# por mercado y subcategoría
bym=pd.DataFrame({'sin':s0.groupby('Mercado').G.sum(),'tope':s1.groupby('Mercado').G.sum(),'v':s0.groupby('Mercado').V.sum()})
OUT['y2015_mkt']={k:dict(sin=float(v.sin),tope=float(v.tope),ventas=float(v.v)) for k,v in bym.iterrows()}
bys=pd.DataFrame({'sin':s0.groupby('Subcategoria').G.sum(),'tope':s1.groupby('Subcategoria').G.sum(),'lsin':s0.groupby('Subcategoria').PL.sum()/c14.groupby('Subcategoria').w.sum(),'ltope':s1.groupby('Subcategoria').PL.sum()/c14.groupby('Subcategoria').w.sum()})
OUT['y2015_sub']={k:dict(sin=float(v.sin),tope=float(v.tope),lsin=float(v.lsin),ltope=float(v.ltope)) for k,v in bys.iterrows()}
# banda: ganancia con el pronóstico bajo/alto (margen igual)
mg0=s0.G.sum()/s0.V.sum();mg1=s1.G.sum()/sum(FC)
OUT['y2015']['band_sin']=[mg0*sum(LO),mg0*sum(HI)];OUT['y2015']['margin_sin']=mg0
OUT['y2015']['margin_tope']=float(s1.G.sum()/s1.V.sum())
OUT['informe']=dict(sin=587493.36,tope100=1000847.69,tope0=913649.57)
for k in ['amp','val','val_tope','y2015','informe']: print(k, OUT[k])

## Límites
Es una foto con los datos de Global Superstore (muestra didáctica). Se probó con un solo año. La pérdida es una asociación, no una causa, y no hay datos de elasticidad: las cifras de la prueba hacia atrás no miden cómo reaccionarían los clientes. En este dataset EMEA y África solo dan descuentos de 0 % o de 60–70 %, y Canadá nunca da descuento, por lo que no hay datos de esos mercados en el rango de 1 % a 20 %.